# 🧬 02 · 多重序列比对（MSA）与保守性分析

> 有了两两比对，下一步是**多条序列一起比**（MSA），寻找**保守位点**——
> 这是功能研究（哪些残基重要）、建进化树、引物设计的基础。
> 本节：MSA 概念 → 一致性序列 → 保守性打分（位点熵）→ 蛋白家族。

## 📋 本节要解决的问题

- 为什么需要 MSA？一条新序列怎么归到一个家族？
- 一致性序列（consensus）怎么算？
- 位点保守性怎么度量？（熵/百分比）
- 保守位点在功能和进化上意味着什么？

## 1️⃣ MSA 与一致性序列

**MSA**：把多条序列按列对齐，让同源位点在同一列。

**一致性序列**：每列取出现最多的碱基/氨基酸组成的代表序列。
> 例：3 条序列第 1 列都是 A → 一致性 A；第 2 列 G/A/G → 多数 G。

In [1]:
import numpy as np

def consensus(seqs):
    "多序列 -> 一致性序列（逐列取众数字符）"
    n = len(seqs)
    L = len(seqs[0])
    cons = []
    for j in range(L):
        col = [s[j] for s in seqs]
        from collections import Counter
        cons.append(Counter(col).most_common(1)[0][0])
    return ''.join(cons)

seqs = ['ACG-TAC', 'ACGGTAC', 'ACGT-AC']
cons = consensus(seqs)
print('MSA 输入:')
for s in seqs: print('  ', s)
print('一致性  :', cons)
assert len(cons) == len(seqs[0])
print('✅ 一致性序列计算完成')

MSA 输入:
   ACG-TAC
   ACGGTAC
   ACGT-AC
一致性  : ACG-TAC
✅ 一致性序列计算完成


## 2️⃣ 位点保守性：百分比一致与香农熵

对每一列：
- **百分比一致**：最多字符的出现比例（简单直观）
- **香农熵**：$H = -\sum_i p_i \log_2 p_i$（0=完全保守，越大越多样化）

> **面试**：为什么用熵而不用百分比一致？
> → 熵能区分「A/G 各半」和「A/C/G/T 各 1/4」，百分比一致只看最大值。

In [2]:
def site_entropy(seqs, j):
    "第 j 列的香农熵"
    from collections import Counter
    col = [s[j] for s in seqs if s[j] != '-']
    if not col:
        return 0.0
    c = Counter(col)
    n = len(col)
    H = 0.0
    for k in c.values():
        p = k / n
        H -= p * np.log2(p)
    return H

def pct_identity(seqs, j):
    col = [s[j] for s in seqs if s[j] != '-']
    if not col:
        return 0.0
    from collections import Counter
    return Counter(col).most_common(1)[0][1] / len(col)

print('第 0 列(全A): 熵=%.3f 一致=%.2f' % (site_entropy(seqs, 0), pct_identity(seqs, 0)))
print('第 3 列(T/-/T): 熵=%.3f 一致=%.2f' % (site_entropy(seqs, 3), pct_identity(seqs, 3)))
assert abs(site_entropy(seqs, 0)) < 1e-9          # 完全保守 → 熵 0
assert abs(pct_identity(seqs, 0) - 1.0) < 1e-9    # 全一致 → 1.0
print('✅ 保守性度量正确（保守位点熵低）')

第 0 列(全A): 熵=0.000 一致=1.00
第 3 列(T/-/T): 熵=1.000 一致=0.50
✅ 保守性度量正确（保守位点熵低）


## 3️⃣ 手算示例：一列的香农熵

> 某列 4 条序列：A, A, G, T（去掉空位）

**字符频率**：A=2/4=0.5，G=1/4=0.25，T=1/4=0.25

**熵**：
$$H = -(0.5\log_2 0.5 + 0.25\log_2 0.25 + 0.25\log_2 0.25)$$
$$= -(-0.5 - 0.5 - 0.5) = 1.5 \text{ bit}$$

> 对比：若全为 A（p=1），H=0（完全保守）；若 4 种碱基各 1/4，H=2 bit（最大）。

In [3]:
# 验证手算
from math import log2
pA, pG, pT = 0.5, 0.25, 0.25
H = -(pA*log2(pA) + pG*log2(pG) + pT*log2(pT))
print('熵(手算=1.5) =', H)
assert abs(H - 1.5) < 1e-9
print('✅ 手算香农熵验证通过')

熵(手算=1.5) = 1.5
✅ 手算香农熵验证通过


## 4️⃣ 蛋白家族的保守性含义

- 保守位点 → 结构关键（催化残基、结合位点）、进化压力强
- 多变位点 → 表面残基、免疫逃逸、可替换区域
- 应用：从保守序列设计简并引物、预测功能位点、判别家族成员

In [4]:
# 模拟一个蛋白保守区：前 3 个位点高度保守，后 2 个可变
motif_seqs = ['YCRKC', 'YCRKM', 'YCRKV', 'YCRRT']
print('motif 比对：')
for s in motif_seqs: print('  ', s)
Hs = [round(site_entropy(motif_seqs, j), 3) for j in range(5)]
print('各列熵:', Hs, '→ 前 3 列低熵 = 保守核心（锌指 Y-C-R-K）')
assert Hs[0] == 0 and Hs[1] < Hs[3]
print('✅ 保守区识别完成（低熵=保守）')

motif 比对：
   YCRKC
   YCRKM
   YCRKV
   YCRRT
各列熵: [0.0, 0.0, 0.0, 0.811, 2.0] → 前 3 列低熵 = 保守核心（锌指 Y-C-R-K）
✅ 保守区识别完成（低熵=保守）


## ✅ 自测清单

- [ ] 能解释 MSA 与一致性序列
- [ ] 能手写位点百分比一致与香农熵
- [ ] 能手算一个小列的熵
- [ ] 能解释保守位点的功能与进化含义